# Evasion Attacks on a Fraud Model

## Poisoning Attack #1

<p align="center">
<img src="./screens/Poisoning-Attack.png" alt="Screenshot" width="700">
</p>

## Model Tampering #2

<p align="center">
<img src="./screens/Model-Tampering.png" alt="Screenshot" width="700">
</p>

## Supply Chain Attack #3

<p align="center">
<img src="./screens/Supply-Chain-Diagram.png" alt="Screenshot" width="700">
</p>

## Evasion Attack #4 (Today)

<p align="center">
<img src="./screens/Evasion-Attack.png" alt="Screenshot" width="700">
</p>

## Intro


**Story.** FinCo's fraud model is trained

- ✅ on clean data (no poisoning),
- ✅ stored and loaded correctly (no tampering),
- ✅ and built from trusted suppliers (no supply chain compromise).

Everything till now has gone right. An attacker can still beat the model, at the moment it is used.

**Evasion attack:**

An evasion attack crafts one input, at the moment it's submitted, so an already-correct, already-trained model misclassifies it. Note that nothing about the model or its training is touched, only the input is changed, usually by a small amount in just the right direction.

 The attacker crafts one transaction, carefully, so that *this specific model* calls it "legit" even though it is fraud. The model itself is never touched. Only the input is.


| Lab | When the attack happens | What is attacked |
|:---:|---|:---|
| Poisoning | During training | The training data |
| Model Tampering | After training, before serving | The stored model file |
| Supply Chain | Anywhere upstream | A supplier (package, model, data, pipeline) |
| **🚩`Evasion`** | **`At prediction time`** | **`One input, sent to an already-correct model`** |



**Learning goals.** After this lab you can

- Craft a **white-box evasion attack** and read its cost/success trade-off.
- Explain why an attacker who can only query the model (no gradients) can still succeed.
- Say what each defense buys you and what it does not.

### Book's Table of content
<p align="center">
<img src="./screens/table-of-contents.png" alt="Screenshot" width="700">
</p>

<p align="center">
<img src="./screens/Book.png" alt="Screenshot" width="700">
</p>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

SEED = 42
np.random.seed(SEED)

## Part-0. The clean, correct model


Same kind of fraud model as the earlier labs: a logistic regression on 10 transaction features, with the fraud class weighted 3 times higher. Nothing here is poisoned, tampered or supply-chain compromised. This is the honest, deployed model the attacker now faces.

In [ ]:
feature_names = ["amount", "hour", "merchant_risk", "device_risk", "country_risk",
                 "velocity", "account_age", "n_prev_fraud", "distance", "channel_risk"]

X, y = make_classification(n_samples=1500, n_features=10, n_informative=6, n_redundant=2,
                           weights=[0.90, 0.10], class_sep=2.0, flip_y=0.0, random_state=SEED)
X = MinMaxScaler().fit_transform(X)   # every feature lives in [0, 1]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)

model = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 3}).fit(X_tr, y_tr)

def report(m, Xe, ye, label=""):
    p = m.predict(Xe)
    print(f"{label:<28}accuracy={accuracy_score(ye, p):.3f}  precision={precision_score(ye, p, zero_division=0):.3f}  recall={recall_score(ye, p, zero_division=0):.3f}")

report(model, X_te, y_te, "clean model, clean data")

# The attacker's target: real fraud transactions in the test set. Each one is already
# correctly flagged. The goal is to change that, one transaction at a time.
fraud_te = X_te[y_te == 1]
print(f"\nfraud rows available to the attacker in the test set: {len(fraud_te)}")

## Part-1. A white-box attack: the Fast Gradient Sign Method (FGSM)


**White-box** means the attacker has the model: its weights, and so its gradient. FGSM asks a direct question: "which direction, in feature space, makes this model's fraud score drop fastest?" and takes one step in that direction.


`eps` controls the size of the step. We sweep it and record, for each value: how many fraud transactions now get called "legit" (**evasion rate**), and how far the attacker had to move each one.

| | What moves | When | Goal |
|---|---|---|---|
| **Poisoning lab, Attack B** | Points added to training data | Before training | Make the *retrained* model worse at validation time |
| **Evasion lab, FGSM** | One already-submitted transaction | At prediction time | Make the *already-trained* model misclassify it now |

<div dir="rtl" style="text-align: right;">

این سلول (در ادامه) یک **حمله FGSM (White-Box)** را در چند مرحله با قدرت‌های مختلف (`eps`) اجرا می‌کند و نتیجه را می‌سنجد:

1. **تولید داده تقلبی (`adv`):** برای هر `eps`، با استفاده از گرادیان مدل (`art_clf`)، تراکنش‌های تقلبی (`fraud_te`) را طوری تغییر می‌دهد که مدل آن‌ها را «مجاز (0)» تشخیص دهد (اگر `eps=0` باشد، داده اصلی را برمی‌گرداند).
2. **سنجش موفقیت (`evaded`):** بررسی می‌کند چه درصدی از این تراکنش‌های تغییر یافته، مدل را فریب داده و «مجاز» شناسایی شده‌اند.
3. **سنجش هزینه (`dist`):** میانگین فاصله اقلیدسی (L2) که هر تراکنش از حالت اصلی خود جابه‌جا شده را محاسبه می‌کند تا نشان دهد مهاجم چه بهایی (تغییر چقدر فاصله) برای فریب مدل پرداخته است.

 </div>


<div dir="rtl" style="text-align: right;">


وقتی مهاجم یک تراکنش را دستکاری می‌کند تا مدل آن را «مجاز» ببیند، نمی‌تواند همه‌چیز را الکی و بی‌نهایت تغییر دهد. باید از حالت اصلی خیلی دور نشود (چون ممکن است لو برود یا سیستم‌های دیگر بانک متوجه شوند).

**فاصله (`dist`) یعنی همان هزینه:**
- اگر تراکنش را خیلی کم تغییر دهد، فریب نمی‌خورد (موفقیت کم).
- اگر بخواهد حتمن فریب بخورد (موفقیت ۱۰۰٪)، باید تراکنش را خیلی زیاد تغییر دهد (هزینه یا فاصله زیاد).

پس **`dist` (تغییر فاصله از حالت اصلی) = بهایی است که مهاجم برای فریب مدل می‌پردازد.** هرچه این عدد بزرگ‌تر باشد، تراکنش دستکاری‌شده غیرواقعی‌تر و قابل‌شناسایی‌تر می‌شود.

 </div>

In [ ]:
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification.scikitlearn import ScikitlearnLogisticRegression

# ART needs a wrapper that knows how to compute gradients through this specific model type.
art_clf = ScikitlearnLogisticRegression(model=model, clip_values=(0.0, 1.0))

#🚩 Craft adversarial versions of X_attack that the model should call class 0 (legit).
def targeted_fgsm(estimator, X_attack, eps, norm=np.inf):
    target = np.zeros((len(X_attack), 2))   # one-hot for class 0, required by ART's targeted mode
    target[:, 0] = 1
    attack = FastGradientMethod(estimator=estimator, eps=eps, targeted=True, norm=norm)
    return attack.generate(X_attack, y=target) # 👈🚩 this is the brain of this function

epsilons = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
sweep = []

for eps in epsilons:
    adv = fraud_te if eps == 0.0 else targeted_fgsm(art_clf, fraud_te, eps)
    evaded = (model.predict(adv) == 0).mean()
    dist = np.linalg.norm(adv - fraud_te, axis=1).mean()# average Euclidean distance each transaction had to move.
    
    sweep.append((eps, evaded, dist))
    print(f"eps={eps:<5} evasion_rate={evaded:.3f}  mean_distance={dist:.3f}")

<div dir="rtl" style="text-align: right;">

**مهاجم به داده‌های واقعی تقلب بانک دسترسی ندارد.**

**کاری که مهاجم در واقعیت انجام می‌دهد:**
 از همان تراکنش‌هایی که قصد ارسالش را دارد شروع می‌کند (او تقلب خودش را می‌شناسد)  یا از داده‌های جایگزین  (یا حتی تصنعی)  شبیه به تقلب استفاده می‌کند، و روی همان‌ها اختلال 
 (perturbation)  را بهینه می‌کند.
 </div>

> Bigger steps (eps) evade more fraud detections, but push the transaction further from the original (mean_distance). For example, `eps=0.1   evasion_rate=0.833  mean_distance=0.316` means that 

- 83.3% of fraud now gets called `legit`, by moving each transaction an average distance of 0.316.

### Two Important Norms in this attack (guys are mathematicians 😎😉)

In math, the $L^p$ norm of a vector $v$ is:

$$
\|v\|_p = \left( \sum_i |v_i|^p \right)^{1/p}
$$

The $L^\infty$ norm is defined as:

$$
\|v\|_\infty = \max_i |v_i|
$$

So passing `np.inf` means: **"measure the size of the change by the biggest single feature change."**


In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot([s[0] for s in sweep], [s[1] for s in sweep], "o-", color="tab:red", label="evasion rate")
ax1.set_xlabel("eps (attack budget)"); ax1.set_ylabel("fraction of fraud called 'legit'", color="tab:red")
ax2 = ax1.twinx()
ax2.plot([s[0] for s in sweep], [s[2] for s in sweep], "s--", color="tab:blue", label="mean distance moved")
ax2.set_ylabel("mean Euclidean distance", color="tab:blue")
plt.title("FGSM: bigger steps evade more, but move the transaction further")
fig.tight_layout(); plt.show()

**Reading the result:**

At `eps=0` the rows are unchanged, so the small evasion rate you see there is just the model's own baseline miss rate (it was never perfect). As `eps` grows, evasion rises steadily rather than jumping sharply, because this is a linear model: there is no single point where the attack "suddenly" works, just a line that gets easier to cross with a bigger step. By `eps=0.2`, every fraud transaction in this batch evades.

## Part-2. Just a change in NORM and be imperceptible


FGSM as used above moves **every feature by the same amount** (just in different directions), because it only uses the **sign** of the gradient, not its size. A different attack, using the actual gradient size (an $L^2$-norm attack) concentrates the change on the features the model relies on most.

We compare the two at a matched outcome (roughly similar evasion rate), feature by feature, sorted by `|coef_|` (the same weight magnitude from the Model Tampering lab).

In [ ]:
adv_sign = targeted_fgsm(art_clf, fraud_te, eps=0.1, norm=np.inf)
adv_l2   = targeted_fgsm(art_clf, fraud_te, eps=0.3, norm=2) 

diff_sign = np.abs(adv_sign - fraud_te).mean(axis=0)
diff_l2   = np.abs(adv_l2 - fraud_te).mean(axis=0)
coef = model.coef_[0] # the 10 weights, one per feature, as a flat 1-D vector.
order = np.argsort(-np.abs(coef))

print(f"evasion: sign-based = {(model.predict(adv_sign)==0).mean():.3f}   size-based (L2) = {(model.predict(adv_l2)==0).mean():.3f}\n")

print(f"{'feature':<16}{'|coef|':>8}{'sign-based':>12}{'size-based':>12}")
for j in order:
    print(f"{feature_names[j]:<16}{abs(coef[j]):>8.2f}{diff_sign[j]:>12.3f}{diff_l2[j]:>12.3f}")

**The point.** The sign-based attack changes `n_prev_fraud` by almost as much as `merchant_risk`. The size-based attack barely touches `n_prev_fraud` and spends almost all its budget on `merchant_risk` and `channel_risk`. Both attacks evade the model. 

An $L^\infty$-style attack is easier to spot as "everything jiggled by the same amount," while an $L^2$-style attack hides by only moving high-impact features.

For feature `j`, `coef[j]` is the weight `w_j` in the decision function:

$$
\text{score} = w_0 x_0 + w_1 x_1 + \dots + w_9 x_9 + b
$$

- **`coef[j] > 0`** → raising feature `j` pushes toward **fraud** (class 1), since the scores get bigger than 0.
- **`coef[j] < 0`** → raising feature `j` pushes toward **legit** (class 0).
- **`|coef[j]|` large** → feature `j` has **big influence** on the decision.

## Part-3. A black-box attack: no gradient, only a yes/no answer


Real attackers often cannot see the model at all. They can only **send a transaction and observe the decision** (approved or flagged); a *decision-based* black-box attack. `HopSkipJump` does exactly this: it never touches a gradient, only `model.predict`.

We count every call to `predict`, the way a real monitoring system would count API calls.


<div dir="rtl" style="text-align: right;">

مهاجم‌های واقعی معمولن اصلن نمی‌تونن مدل رو ببینن. تنها کاری که می‌تونن بکنن اینه که **یک تراکنش بفرستن و تصمیم مدل رو ببینن** (تایید شد یا مسدود شد) ؛ به این می‌گن حمله‌ی *جعبه‌سیاه مبتنی بر تصمیم* (decision-based black-box). الگوریتم `HopSkipJump` دقیقن همین کار رو می‌کنه: هیچ‌وقت به گرادیان (مشتق) دست نمی‌زنه و فقط از `model.predict` استفاده می‌کنه.

ما هر بار صدا زدن `predict` رو می‌شماریم، همون‌طور که یک سیستم نظارتی واقعی تعداد درخواست‌های API رو می‌شمره.
 </div>


In [ ]:
from art.attacks.evasion import HopSkipJump

# This wrapper remembers how many rows it was asked to score.
class CountingLR(ScikitlearnLogisticRegression):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.n_queries = 0
    def predict(self, x, **kw):
        self.n_queries += len(x)
        return super().predict(x, **kw)

counting_clf = CountingLR(model=model, clip_values=(0.0, 1.0))
# clip_values=(0.0, 1.0) tells the attack: "The features in this dataset only live between 0 and 1.
# Never push them outside this range."

probe_rows = fraud_te[:10]
# Here we give the attacker the 10 Fraud rows. But in a real black-box attack, the attacker would use their own
# fake fraud transactions (the ones they actually want to submit) and ask the model to score those.

hsj = HopSkipJump(classifier=counting_clf, targeted=True, max_iter=10, max_eval=200, init_eval=20, verbose=True)

adv_bb = hsj.generate(probe_rows, y=np.zeros(len(probe_rows)))
# This is the line that actually runs the black-box attack.
# It takes the 10 fraud rows, asks the model hundreds of questions to find the blind spots,
# and returns adv_bb; the modified, "legit-looking" versions of those fraud transactions.
# Indeed, hsj.generate() returns 10 new rows (the same shape as probe_rows),
# but they have been carefully tweaked (perturbed) to be "bad data."

evaded_bb = (model.predict(adv_bb) == 0).mean()
print(f"black-box evasion rate: {evaded_bb:.3f}")
print(f"total prediction queries used: {counting_clf.n_queries}  (about {counting_clf.n_queries / len(probe_rows):.0f} per transaction)")

Based on the output `total prediction queries used: 5583  (about 558 per transaction)`, for each single row, it asked the model 558 questions (queries).

So HSJ does this loop thousands of times:

- "Hey model, what about this row?" (Query #1)

- Model: "Fraud."

- "Okay, what if I change the amount by +0.05?" (Query #2)

- Model: "Still Fraud."

- "What if I change merchant_risk by -0.1?" (Query #3)

- Model: "Legit."

It takes those tiny changes, builds a `map` of the model's decision boundary, and steps across it.

**The point.** Without ever seeing a weight or a gradient, the attacker still evades almost every transaction, just by watching the model's decisions and adjusting. The cost is **queries**, not skill: about 500 predictions per evaded transaction here. That cost is also the opening for a defense (we will see in the sequl): a real user never triggers 500 near-identical fraud checks in a row.

<div dir="rtl" style="text-align: right;">

**یک جمع‌بندی از این دو حمله‌ی آخر**
👇👇👇👇

</div>

**White-box (FGSM):**

Fast, perfect, but requires stealing the model weights.

**Black-box (HopSkipJump):**

Slower and louder (makes lots of queries), but works without any internal access. It proves that hiding the model weights is not a complete defense.

## Part-4. Defenses


### Part-4: Adversarial training
Take the training data, add some adversarial fraud examples to it (still labeled "fraud"), and retrain the model on the combined set.

**The idea:** the model learns that the region just past the boundary is still fraud.

<div dir="rtl" style="text-align: right;">

**تیم دفاع کننده باید بنوعی در آموزش، مدل را با داده‌های خصمانه آموزش دهد که مدل در محیط عملیاتی براحتی توسط حمله‌کنندگان گول نخورد.**

مدافع نیازی به تراکنش واقعیِ ساخته‌شده توسط مهاجم ندارد. مدافع نمونه‌های خصمانه‌ی خودش را با استفاده از مدل خودش و داده‌های تاریخی تقلب خودش، از طریق شبیه‌سازی حمله به‌دست می‌آورد.
 </div>

In [ ]:
fraud_tr = X_tr[y_tr == 1]
target_tr = np.zeros((len(fraud_tr), 2)); target_tr[:, 0] = 1
adv_tr = targeted_fgsm(art_clf, fraud_tr, eps=0.1)   # crafted on TRAIN data only

X_aug = np.vstack([X_tr, adv_tr])
y_aug = np.concatenate([y_tr, np.ones(len(adv_tr))])   # still labeled fraud
model_robust = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 3}).fit(X_aug, y_aug)
art_robust = ScikitlearnLogisticRegression(model=model_robust, clip_values=(0.0, 1.0))

report(model_robust, X_te, y_te, "adversarially trained")

print(f"\n{'eps':<8}{'baseline evasion':>18}{'robust evasion':>16}")
for eps in [0.05, 0.1, 0.2, 0.3]:
    base_ev = (model.predict(targeted_fgsm(art_clf, fraud_te, eps)) == 0).mean()
    rob_ev  = (model_robust.predict(targeted_fgsm(art_robust, fraud_te, eps)) == 0).mean()
    print(f"{eps:<8}{base_ev:>18.3f}{rob_ev:>16.3f}")

<div dir="rtl" style="text-align: right;">

**نتیجه‌ی صادقانه:**

 آموزش خصمانه تقریبن کمکی نکرد. حتی در مواردی بدتر هم شد.

 </div>


### Part-4: Monitoring the thing the attack actually costs: queries
Part 3 showed the black-box attack needs roughly 500 predictions per evaded transaction. A legitimate transaction needs exactly one.

<div dir="rtl" style="text-align: right;">

**در برخی از مواقع دفاع بسیار راحت است**

برای اپ و استک‌مون محدودیت ارسال درخواست بزاریم. دقت کنید که حتمن باید دوجا اینرو اعمال کنیم. هم در سطح پروکسی و هم در سطح اپ.
 </div>

In [ ]:
# query_log: {account_id: number of prediction calls in the last hour}
def flag_accounts(query_log, window_limit=200):
    return {acct: n for acct, n in query_log.items() if n > window_limit}

normal_traffic = {"acct_1001": 1, "acct_1002": 3, "acct_1003": 1}

attacker_traffic = {**normal_traffic, "acct_9999": counting_clf.n_queries}

print("flagged for review:", flag_accounts(attacker_traffic))

In [ ]:
print({**normal_traffic, "A": 1})

**The point.** A rate limit of, say, 20 fraud-check queries per account per hour costs legitimate users nothing and would have caught `acct_9999` immediately. This does not stop a *patient* attacker who spreads queries across many accounts or many hours, or one who has only a few attempts to begin with (a one-shot FGSM attack generates no unusual query volume at all). It is one more layer, aimed specifically at the query-heavy black-box style of attack from Part 5, not a general fix.


This defense doesn't look at the data at all. It doesn't care what the attacker sent, only how many times. That's exactly the thing Part 5 revealed: a black-box attack needs hundreds of queries per evaded transaction, while a real user needs one. So counting queries is a cheap, effective way to catch this specific style of attack.

> **Question:**
>
> Where in our architecture we should put the logic for checking user's limits? on app or on proxy?
>
> **Answer**:
> 
> BOTH. proxy for volume and app for meaning. Relying on the proxy alone would miss this specific attack, because the proxy doesn't know these are fraud-model queries at all.

## Final Notes

<div dir="rtl" style="text-align: right;">

1- هکرها از هر روشی که بتوانند استفاده می‌کنند تا اطلاعاتی در خصوص مدل و دیتاست که با اون مدل رو آموزش دادیم، بدست بیاورند. به هیچ وجه راجع به کاری که دارین انجام میدن جایی پست نزارید و به کسی هم اعتماد نکنید که جزئیات رو بگید.

2- ما حملاتی داریم که بشدت غیرقابل شناسایی هستند، مثلن
`Jacobian-based Saliency Map Attack (JSMA)`.
این روش‌ها نیاز به منابع محاسباتی بالایی دارند و عمومن هکرهای سازمان یافته از اونها استفاده می‌کنند. 

3- بعضی وقتها هم همین بزرگان امنیتی خودمون آش پرچربی برامون پختند. مثلن همین جناب نیکولاس کارلینی روشی رو به دنیا معرفی کرد که با نام
`Carlini and Wagner (C&W)`
اونرو می‌شناسند. این روش بهینه شده که کمترین تغییر برای عوض کردن دسته‌بندی رو انجام دهد. یعنی با کمترین میزان اختلال یا پرتوربیشن میشه مدل رو گول زد.
حالا تیم‌های دفاعی باید بشینند و چاره‌ای بیاندیشند برای این موارد که مقابله باهاشون بسیار بسیار سخته 😎.
 </div>

> `C&W` is a benchmark for robustness. A defense that stops `FGSM` is not impressive if it fails against `C&W`. The most reliable path is Adversarial Training.

<figure style="text-align: center;">
  <img src="./screens/Carlini.jpg" alt="Screenshot" width="400">
  <figcaption>
    Nicholas Carlini is an American researcher affiliated with Anthropic and previously with Google DeepMind who has published research in the fields of computer security and machine learning. He is known for his work on adversarial machine learning, particularly his work on the Carlini & Wagner attack in 2016.
  </figcaption>
</figure>